# 3-1회차 | 전처리(Preprocessing)

**핵심 질문**: 데이터를 그대로 넣으면 안 되는 이유는?

**오늘의 목표**
1. **인코딩**: 글자(문자열)는 모델이 이해 못 한다 → 숫자로 바꿔주기
   - Label vs One-Hot vs Ordinal — 언제 뭘 쓰는가
2. **스케일링**: 모델에 따라 값의 크기 차이가 영향을 줄 수 있다 → 크기 맞춰주기
   - StandardScaler vs MinMaxScaler vs RobustScaler
3. **정규화**: 각 데이터의 길이를 똑같이 맞춰주는 방법 (방향만 중요할 때)
4. **데이터 누수**: 평가용으로 남겨둔 데이터를 미리 쓰면 안 된다 → 훈련 데이터로만 fit
   - Pipeline으로 **fit 범위를 관리하기 쉽게**

---
## 3회차에서 전처리를 다루는 이유

**2회차에서 모델을 선택하는 기준을 배웠음**
하지만 좋은 모델을 골라도, **데이터가 제대로 준비되지 않으면 성능이 안 나옴**

![최고의 셰프도 상한 재료로는 요리할 수 없습니다](스크린샷%202026-02-08%20오전%209.40.41.png)

![모델을 위한 재료 손질: 전처리](스크린샷%202026-02-08%20오전%209.41.06.png)

### 전처리가 필요한 이유

| 문제 | 결과 | 해결책 |
|------|------|--------|
| 문자열 데이터 | 모델이 계산 불가 | **인코딩** |
| 값의 크기 차이(단위/범위) | 거리 기반 모델에서 큰 단위 피처가 거리 계산을 지배 | **스케일링** |
| 길이(크기)보다 방향이 중요 | 의미/패턴 비교가 어려움 | **정규화** |
| 전처리를 전체 데이터로 fit | 평가가 무효(누수) | **Pipeline** |

> **3회차 핵심 문장:**
> "모델은 숫자만 이해하고, **모델에 따라** 값의 크기 차이가 학습에 영향을 줄 수 있음."

---
## Part 1. 인코딩 (Encoding)

### 인코딩이란?

- 모델은 글자(문자열)를 직접 계산할 수 없음
- 예: 색깔 = "빨강(red)", "초록(green)", "파랑(blue)"
- 글자를 그대로 넣으면 모델은 "이게 뭐지?" 하고 멈춤

> 마치 한국어 시험지를 원어민 영어 선생님에게 가져가면, 선생님은 못 알아보는 것과 같음
> 그래서 "숫자"로 번역(인코딩)을 해줘야 함

In [1]:
import pandas as pd

data = pd.DataFrame({"color": ["red","green","blue","red","green","blue"]})
data

,color
0,red
1,green
2,blue
3,red
4,green
5,blue


### 레이블 인코딩 vs 원-핫 인코딩

1) **레이블 인코딩(Label Encoding)**
   - 각 색깔에 0, 1, 2 같은 번호를 붙이기
   - (실제 부여되는 번호는 라이브러리가 정함. 아래에서 직접 확인)
   - 문제: 0, 1, 2로 번호를 붙이면 **0 < 1 < 2 라는 순서**가 데이터에 생김
   - 모델은 그 순서를 **의심하지 않고 그대로 사용**함 (뒤에서 직접 확인)

2) **원-핫 인코딩(One-Hot Encoding)**
   - red = [1,0,0]
   - green = [0,1,0]
   - blue = [0,0,1]
   - 순서를 만들지 않고, 단지 "있다/없다"만 표시

### 참고: 차원(Dimension)이 뭔가요?

- **차원 수**란, 모델이 한 데이터를 판단할 때 **참고하는 입력값의 개수**
- 실습에서는 보통 **"모델에 들어가는 컬럼의 개수"**라고 이해하면 됨

> **예시**
> - 원래 color 1개 컬럼 → 차원 = 1
> - 원-핫 인코딩 후 red, green, blue 3개 컬럼 → 차원 = 3

- 원-핫 인코딩처럼 전처리를 하면 모델이 데이터를 더 잘 이해할 수 있도록 **차원이 늘어날 수 있음**
- 하지만 차원이 늘수록 **데이터 구조는 더 복잡해지고**, 중요한 정보와 중요하지 않은 정보가 섞일 수 있음
- 그래서 이후에는 **차원을 줄이거나 중요한 축만 남기는 방법**도 필요해짐 (나중에 배움)

In [2]:
from sklearn.preprocessing import LabelEncoder, OneHotEncoder

# 레이블 인코딩
le = LabelEncoder()
data["color_label"] = le.fit_transform(data["color"])

# 원-핫 인코딩
# -> 각 카테고리를 새로운 컬럼으로 분해
# -> 데이터 개수(행)는 그대로, 입력 차원(컬럼)만 증가
ohe = OneHotEncoder(sparse_output=False)
ohe_matrix = ohe.fit_transform(data[["color"]])
ohe_df = pd.DataFrame(ohe_matrix, columns=ohe.categories_[0])

# 합치기
data_ohe = pd.concat([data, ohe_df], axis=1)
data_ohe

,color,color_label,blue,green,red
0,red,2,0.0,0.0,1.0
1,green,1,0.0,1.0,0.0
2,blue,0,1.0,0.0,0.0
3,red,2,0.0,0.0,1.0
4,green,1,0.0,1.0,0.0
5,blue,0,1.0,0.0,0.0


- "color_label" 열은 0,1,2로 번호를 붙인 것 (순서 착각 위험)
- "blue, green, red" 열은 원-핫 인코딩 결과 (**임의의 순서를 만들지 않음**)

> 실제로는 **원-핫 인코딩**을 많이 씁니다.

### 레이블 인코딩의 문제점 — 선형 모델에서 확인(모델은 나중에 배울 예정이니 그런가보다 하셈)

레이블 인코딩으로 붙인 숫자에는 **크기 순서**가 생깁니다.
모델은 그 순서를 의심하지 않고 그대로 씀. 직접 확인해볼 것.

---

### 잠깐 — 실행 전에 예측해볼 것

`red`, `green`, `blue`에 **0, 1, 2 같은 번호만 붙여서** 모델에 넣으면,
모델이 그걸 그냥 **이름표**로 이해할까?

> 예상 먼저 적고 아래 셀 실행할 것.

In [3]:
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
import numpy as np, pandas as pd

np.random.seed(42)
colors = pd.Series(np.random.choice(["red", "green", "blue"], size=100))

# red 또는 blue이면 target=1, green이면 target=0
target = colors.map({"red": 1, "blue": 1, "green": 0})
df_test = pd.DataFrame({"color": colors, "target": target})

# 두 가지 표현 방식
X_label  = LabelEncoder().fit_transform(df_test["color"]).reshape(-1, 1)
X_onehot = OneHotEncoder(sparse_output=False).fit_transform(df_test[["color"]])
y = df_test["target"]

# 같은 모델에 서로 다른 표현을 넣어본다
acc_label  = accuracy_score(y, LogisticRegression().fit(X_label,  y).predict(X_label))
acc_onehot = accuracy_score(y, LogisticRegression().fit(X_onehot, y).predict(X_onehot))

print("=== 로지스틱 회귀 (선형 모델) ===")
print(f"레이블 방식 학습 데이터 정확도: {acc_label:.3f}")
print(f"원-핫 방식 학습 데이터 정확도: {acc_onehot:.3f}")
print(f"\n참고 - 다수 클래스 비율: {y.mean():.3f}")

=== 로지스틱 회귀 (선형 모델) ===
레이블 방식 학습 데이터 정확도: 0.640
원-핫 방식 학습 데이터 정확도: 1.000

참고 - 다수 클래스 비율: 0.640


> **이 숫자는 학습 데이터 정확도임.**
>
> 이걸 그냥 "성능"이라고 부르면 안 되는 이유는 **3-2회차**에서 다룸.
> 지금 보는 건 일반화 성능이 아니라,
> **같은 데이터를 다르게 표현했을 때 이 모델이 그 패턴을 표현할 수 있는가**임.

레이블 방식의 0.640은 우연한 숫자가 아님. **다수 클래스 비율과 정확히 같음.**
모든 입력에 같은 답을 내놨다는 뜻.

근데 모델이 학습을 못 한 걸까? 확인해볼 것.

In [4]:
# 모델이 정말 "학습을 못 한" 것인지 열어서 확인
lr_label = LogisticRegression().fit(X_label, y)

print(f"coef_      : {lr_label.coef_[0][0]:.4f}")
print(f"intercept_ : {lr_label.intercept_[0]:.4f}")
print(f"n_iter_    : {lr_label.n_iter_[0]}  (수렴 완료)")

print("\n입력 숫자별 예측 확률 P(target=1)")
for v, name in enumerate(LabelEncoder().fit(df_test["color"]).classes_):
    p = lr_label.predict_proba([[v]])[0, 1]
    print(f"  {name:6s}({v}) -> {p:.4f}")

coef_      : 0.0458
intercept_ : 0.5288
n_iter_    : 4  (수렴 완료)

입력 숫자별 예측 확률 P(target=1)
  blue  (0) -> 0.6292
  green (1) -> 0.6398
  red   (2) -> 0.6503


### 모델은 우리가 준 숫자의 관계를 그대로 씀

확률이 **blue(0) < green(1) < red(2) 순서로 커지고 있음.**

알파벳 순으로 아무렇게나 붙인 번호를, 모델은 **실제 순서라고 믿고 성실하게 학습**함.
그래서 세 값이 전부 0.5를 넘고, 모든 입력에 1을 예측하게 된 것.

> **모델이 이상하게 학습한 게 아님.**
> 우리가 준 숫자의 관계를 모델이 그대로 쓴 것뿐임.

여기서 `LabelEncoder`가 나쁜 도구라는 결론으로 가면 안 됨.
`LabelEncoder`는 원래 **타깃 `y`를 인코딩하는 도구**이고,
여기처럼 입력 명목형 feature에 쓰면 **임의의 순서가 모델에 전달될 수 있다**는 게 핵심.

In [5]:
from sklearn.tree import DecisionTreeClassifier

acc_label_dt  = accuracy_score(y, DecisionTreeClassifier(random_state=42).fit(X_label,  y).predict(X_label))
acc_onehot_dt = accuracy_score(y, DecisionTreeClassifier(random_state=42).fit(X_onehot, y).predict(X_onehot))

print("=== 결정 트리 (트리 모델) ===")
print(f"레이블 방식 학습 데이터 정확도: {acc_label_dt:.3f}")
print(f"원-핫 방식 학습 데이터 정확도: {acc_onehot_dt:.3f}")
print("\n-> 이번 toy dataset에서는 두 표현 모두 훈련 데이터를 완전히 분리했습니다.")
print("   트리도 숫자의 순서를 기준값으로 사용합니다.")
print("   '트리 모델이면 임의의 정수 인코딩이 항상 괜찮다'는 뜻은 아닙니다.")

=== 결정 트리 (트리 모델) ===
레이블 방식 학습 데이터 정확도: 1.000
원-핫 방식 학습 데이터 정확도: 1.000

-> 이번 toy dataset에서는 두 표현 모두 훈련 데이터를 완전히 분리했습니다.
   트리도 숫자의 순서를 기준값으로 사용합니다.
   '트리 모델이면 임의의 정수 인코딩이 항상 괜찮다'는 뜻은 아닙니다.


### Ordinal(순서형) 인코딩

- **명목형(순서 없음)**: 원-핫 인코딩
- **순서형(작<중<대)**: OrdinalEncoder로 **그 순서를 표현할 수 있음**
  - 다만 Ordinal을 쓴다고 **항상 성능이 더 좋아지는 것은 아님**
  - 장점은 성능이 아니라: 실제 순서를 표현할 수 있고, 입력 차원을 늘리지 않음

> 순서가 **의미 있는** 경우에만 오디널 인코딩을 사용!

In [6]:
import pandas as pd
from sklearn.preprocessing import OrdinalEncoder

# 데이터 프레임 생성
df_ord = pd.DataFrame({"size":["small","medium","large","medium","small"]})
# Ordinal(순서형) 인코더 가지고 범주형 -> 숫자로 인코더
#  categories = 파라미터에 순서를 직접 우리가 지정해줌
# small 0, medium 1, large 2
ord_enc = OrdinalEncoder(categories=[["small","medium","large"]])
# fit_transform : 학습+ 변환 한번에 제가 귀찮아서 자주 애용 아이
df_ord["size_code"] = ord_enc.fit_transform(df_ord[["size"]])
df_ord

,size,size_code
0,small,0.0
1,medium,1.0
2,large,2.0
3,medium,1.0
4,small,0.0


### 인코딩 방법 비교

| 방법 | 설명 | 언제 쓰나 | 주의점 |
|------|------|----------|--------|
| **레이블 인코딩** | 각 범주에 0, 1, 2 같은 번호 부여 | (주로) **타깃 y** 라벨, 또는 **순서형**을 임시로 코드화 | X에 쓰면 **가짜 순서** 위험 (특히 선형/거리 모델) |
| **원-핫 인코딩** | red=[1,0,0], green=[0,1,0] | **순서 없는 명목형 X** (순서를 만들지 않는 출발점) | 차원 증가(카테고리 많으면 폭발) |
| **오디널 인코딩** | small=0, medium=1, large=2 | **순서가 의미 있을 때** | 순서를 직접 지정해야 함 |

### 인코딩 선택 가이드(항상 예외는 있음)

```markdown
범주형 데이터인가?
    ├── 실제 순서가 있는가?
    │       └── YES → OrdinalEncoder로 그 순서를 표현할 수 있음
    │
    └── 순서가 없는가?
            └── 범주 수가 많지 않다면 One-Hot이 흔한 출발점

LabelEncoder → 주로 타깃 y
```

> 실제 선택은 **범주 수, 데이터 크기, 모델**에 따라 달라질 수 있음.

> 방금 확인한 확률 세 줄(blue 0.6292 < green 0.6398 < red 0.6503)이
> 이 표의 "X에 쓰면 가짜 순서 위험"이 실제로 무슨 뜻인지 보여준 것.

### 텍스트 인코딩 — CountVectorizer로 문장을 숫자로 바꾸기

지금까지 색상(red, green, blue)처럼 **짧은 범주형** 데이터를 인코딩했음
그렇다면 **문장(텍스트)**은 어떻게 숫자로 바꿀까??????

- **CountVectorizer**: 문장에서 단어가 **몇 번 나왔는지(빈도)**를 세어 벡터로 변환
- 예: "영화 재밌다" → [1, 0, 1, 0, ...] (단어별 등장 횟수)

> **이 예제는 실제 한국어 감성분석 모델의 성능을 보여주는 것이 아닙니다.**
> CountVectorizer가 학습한 vocabulary로 **새 문장을 어떻게 숫자로 바꾸는지** 확인하는 toy example임.

> 이것도 일종의 인코딩임!
> 글자를 숫자로 바꿔서 모델이 계산할 수 있게 만드는 것.

> 참고: 한국어는 공백 기준 토큰화만으로는 단어가 부정확할 수 있음.
> (실무에서는 형태소 분석기나 서브워드 토크나이저를 쓰기도 함.)

In [7]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression

# 1) 데이터 준비 - 문장 6개
texts = [
    "이 영화 정말 재밌다",
    "완전 감동적이다",
    "너무 지루하고 별로였다",
    "시간 낭비였다",
    "연출이 좋았다",
    "스토리가 엉망이다",
]
labels = [1, 1, 0, 0, 1, 0]   # 1 = 긍정, 0 = 부정

# 2) 벡터화: 단어가 몇 번 나왔는지 세는 도구
vectorizer = CountVectorizer()
X = vectorizer.fit_transform(texts)

print("단어 사전:", vectorizer.get_feature_names_out())

# 3) 모델 학습
model = LogisticRegression()
model.fit(X, labels)

단어 사전: ['감동적이다' '낭비였다' '너무' '별로였다' '스토리가' '시간' '엉망이다' '연출이' '영화' '완전' '재밌다' '정말'
 '좋았다' '지루하고']


,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`mul

### 이제 새 문장을 넣어볼 것

```
"이 영화는 별로다"
```

**예측해볼 것. 모델이 부정(0)으로 맞힐까?**

In [8]:
new_text = ["이 영화는 별로다"]

# 주의: 새 문장은 transform만! (fit 하면 사전이 바뀜)
new_X = vectorizer.transform(new_text)

print("예측 결과:", model.predict(new_X))

예측 결과: [0]


`0`이 나왔음. 부정으로 맞혔네.

> **근데 정말 맞힌 걸까?**

모델 안을 열어볼 것.

In [9]:
import numpy as np

feature_names = vectorizer.get_feature_names_out()
matched = feature_names[new_X.toarray()[0] > 0]

print("사전에서 매칭된 단어 수:", new_X.nnz, "개")
print("매칭된 단어           :", list(matched))
print("예측 확률 [부정, 긍정] :", model.predict_proba(new_X)[0].round(4))
print("예측 결과             :", model.predict(new_X)[0])

사전에서 매칭된 단어 수: 0 개
매칭된 단어           : []
예측 확률 [부정, 긍정] : [0.5 0.5]
예측 결과             : 0


### 아님. 모델은 이 문장을 이해하지 못했음

- 매칭된 단어: **0개**
- 예측 확률: **[0.5, 0.5]** — 완전히 반반. 모델이 **아무 판단도 못 한 상태**임.

`영화는`은 사전의 `영화`와 다른 단어고, `별로다`도 `별로였다`와 다른 단어임.
`이`는 한 글자라 기본 설정에서 아예 단어로 세어지지도 않음.

근데 화면에는 `0`만 찍혔음.

> **출력이 나왔다는 것과, 그 출력에 근거가 있다는 건 다름.**

이 감각은 3-2회차에서 다시 만남.
`정확도가 90%다` → 그럼 진짜 필요한 일을 잘하고 있는 걸까?

### 그럼 사전에 있는 단어를 써볼 것

In [10]:
better_text = ["정말 지루하고 별로였다"]
better_X = vectorizer.transform(better_text)

matched = feature_names[better_X.toarray()[0] > 0]
print("사전에서 매칭된 단어 수:", better_X.nnz, "개")
print("매칭된 단어           :", list(matched))
print("예측 확률 [부정, 긍정] :", model.predict_proba(better_X)[0].round(4))
print("예측 결과             :", model.predict(better_X)[0])

사전에서 매칭된 단어 수: 3 개
매칭된 단어           : ['별로였다', '정말', '지루하고']
예측 확률 [부정, 긍정] : [0.5728 0.4272]
예측 결과             : 0


---

### 먼저, 지난 시간에 못 푼 문제부터

2회차 마지막 표 기억하시나요.

| 모델 | CV 평균 | CV 표준편차 |
|------|---------|------------|
| LogisticRegression | 0.9927 | 0.0089 |
| DecisionTree | 0.9671 | 0.0241 |
| **KNN (k=5)** | **0.7801** | 0.0355 |

KNN만 유독 낮고, 흔들림도 제일 컸음. 이유도 그때 말했었음.

```
bill_length_mm :   32 ~   60      (범위 약   28)
body_mass_g    : 2700 ~ 6300      (범위 약 3600)
```

> 거리를 계산하면 `body_mass_g`의 차이가 숫자상 100배는 크게 반영됨.
> KNN은 사실상 **몸무게만 보고** 이웃을 찾은 셈임.

그리고 이렇게 미뤄뒀었음.

> **이 문제는 오늘 못 고침. 스케일링(전처리)을 배워야 고칠 수 있음.**

**오늘 그 숙제를 받음.**

**[예측]** 숫자 크기 차이가 거리 계산에 어떤 영향을 주는지 한 줄로 적어볼 것.
그다음 아래 예제로 원리부터 확인함.

---
## Part 2. 스케일링 (Scaling)

### 스케일링이란?

- 두 가지 정보가 있음.
  - 학생 키 = [150cm, 160cm, 170cm] → 범위: 20
  - 학생 용돈 = [1000원, 20000원, 50000원] → 범위: 49000

> 거리를 계산하는 모델이라면, 용돈의 차이가 커서 키의 차이가 거의 묻히게 됨
> 마치 수학 점수는 0-100, 음악 점수는 0-1억일 때 → 음악만 중요해지는 상황
> 그래서 범위를 비슷하게 맞춰주는 작업이 필요

In [11]:
import numpy as np
from sklearn.preprocessing import StandardScaler

X = np.array([[150,1000],
              [160,20000],
              [170,50000]])

scaler = StandardScaler()
# 평균 0, 표준편차 1 (-1 ~ 1)
# 모델이 공평하게 학습(비교도 하구)
X_scaled = scaler.fit_transform(X)

df_scale = pd.DataFrame(
    np.hstack([X, X_scaled]),
    columns=["키(cm)","용돈(원)","키(스케일링 후)","용돈(스케일링 후)"]
)
df_scale

,키(cm),용돈(원),키(스케일링 후),용돈(스케일링 후)
0,150.0,1000.0,-1.224745,-1.123698
1,160.0,20000.0,0.000000,-0.181775
2,170.0,50000.0,1.224745,1.305473


- 원래 데이터는 "용돈" 값이 너무 커서 차이가 엄청 큼
- 스케일링 후에는 "키"와 "용돈"이 비슷한 범위로 맞춰짐

> 거리 기반 모델에서 큰 단위의 피처가 거리 계산을 지배하는 문제를 줄일 수 있음

### RobustScaler — 이상치에 강한 스케일러
- 중앙값/사분위 범위(IQR)로 스케일링 → 평균/표준편차 대신 **통계량** 사용

In [12]:
import numpy as np, pandas as pd
from sklearn.preprocessing import RobustScaler, StandardScaler

X = np.array([[1., 10.],
              [2., 12.],
              [3., 13.],
              [100., 1000.]])   # 이상치
# 평균 = 0, 표준편차 1로 변환 (z-Score)
# 평균을 가지고 노니까 이상치에 민감
sc = StandardScaler().fit_transform(X)
# 중앙값, IQR을 가지고 스케일링
# 이상치에 상대적으로 덜 민감함
rb = RobustScaler().fit_transform(X)

df_rb = pd.DataFrame(np.hstack([X, sc, rb]),
                     columns=["x1","x2","Std_x1","Std_x2","Robust_x1","Robust_x2"])
df_rb

,x1,x2,Std_x1,Std_x2,Robust_x1,Robust_x2
0,1.0,10.0,-0.600832,-0.581243,-0.058824,-0.010070
1,2.0,12.0,-0.577270,-0.576570,-0.019608,-0.002014
2,3.0,13.0,-0.553708,-0.574233,0.019608,0.002014
3,100.0,1000.0,1.731810,1.732045,3.823529,3.977845


### MinMaxScaler vs StandardScaler
- MinMax: [0,1] 구간으로 맞춤 (최솟값/최댓값 기반)
- Standard: 평균 0, 표준편차 1 (z-score 표준화)
- 어떤 스케일러가 좋은지는 **데이터 분포/모델**에 따라 다름

In [13]:
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
from sklearn.pipeline import Pipeline
from sklearn.datasets import load_iris

# iris 데이터 로드
X, y = load_iris(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X,y, test_size=0.3, stratify=y, random_state=42)


pipe_minmax = Pipeline([("scaler", MinMaxScaler()), ("knn", KNeighborsClassifier(n_neighbors=5))])
pipe_std    = Pipeline([("scaler", StandardScaler()), ("knn", KNeighborsClassifier(n_neighbors=5))])

pipe_minmax.fit(X_train, y_train); pred_mm = pipe_minmax.predict(X_test)
pipe_std.fit(X_train, y_train);    pred_sd = pipe_std.predict(X_test)

print(f"[MinMax] Acc={accuracy_score(y_test, pred_mm):.3f}, F1(macro)={f1_score(y_test, pred_mm, average='macro'):.3f}")
print(f"[Std   ] Acc={accuracy_score(y_test, pred_sd):.3f}, F1(macro)={f1_score(y_test, pred_sd, average='macro'):.3f}")

[MinMax] Acc=0.933, F1(macro)=0.933
[Std   ] Acc=0.911, F1(macro)=0.910


> **이번 Iris + KNN 실행에서는 MinMax가 조금 높게 관찰됐음.**
>
> 근데 **한 번의 split**만으로 "Iris에는 MinMax가 더 좋다"고 결론내리지 않음.
> 왜 그렇게 나왔는지에 대한 인과도 이 실험만으로는 증명되지 않음.
>
> 여기서 확인할 건 하나임.
> MinMaxScaler는 각 피처를 일정한 범위로 바꿔,
> **단위·범위 차이가 거리 계산을 지배하는 문제를 줄여줌.**

### 핵심
- 피처들의 단위·범위 차이가 큰 KNN에서는 스케일링이 특히 중요
- 어느 스케일러가 더 나은지는 **데이터와 모델에 따라 달라짐**. 한 번의 결과로 정하지 않음

### 스케일러 비교

| 스케일러 | 방식 | 특징 | 언제 쓰나 |
|---------|------|------|----------|
| **StandardScaler** | 평균=0, 표준편차=1 (z-score) | 많은 모델에서 안정적으로 잘 작동 | 기본 선택(선형/거리 기반에서 자주) |
| **MinMaxScaler** | [0, 1] 구간으로 변환 | 이상치(outlier)에 민감 | 범위가 중요한 값(픽셀 등) / 데이터가 깔끔할 때 |
| **RobustScaler** | 중앙값/IQR 기반 | 이상치에 강함 | 이상치가 있을 때 |

### 스케일러 선택의 출발점 — 절대 규칙 아님

```
이상치의 영향을 줄이고 싶은가?        → RobustScaler 후보
0~1 같은 특정 범위가 필요한가?        → MinMaxScaler 후보
일반적인 비교를 시작하려는가?          → StandardScaler 후보
```

> 세 개 중 하나 고르고 끝내는 게 아니라, **후보를 정해서 비교해보는 것**이 출발점임.

---
## Part 3. 정규화 (Normalization)

### 정규화란?
- 앞에서 CountVectorizer로 만든 것도 전부 **벡터(숫자 배열)**이었음
- 이제부터는 벡터를 이렇게 해석합세:
  - **길이(크기)**: “얼마나 많이/크게” (예: 문장 길이, 단어 총량)
  - **방향(패턴)**: “어떤 조합/비율인가” (예: 어떤 단어들이 함께 나왔는가)

> 정규화의 목적: **길이 영향은 지우고, 방향(패턴)만 비교**

정규화는 각 데이터 벡터의 길이를 1로 맞추는 것.  
예: [3,4] → 길이=5 → [0.6,0.8]

> 마치 지도에서 "집까지의 거리"는 무시하고, "집이 어느 방향에 있는지"만 보는 것과 같음

**음악 취향 비유:**
- A는 록 100곡+재즈 50곡, B는 록 10곡+재즈 5곡
- → 절대 개수(길이)는 다르지만, 비율(방향)은 동일
- → 정규화하면 같은 취향 패턴으로 인식됨


#### 정리
- 스케일링은 **피처(열) 기준**, 정규화는 **샘플(행) 기준**
- 정규화는 **방향 비교**가 핵심 (텍스트/추천/임베딩에서 자주 사용)

In [14]:
from sklearn.preprocessing import Normalizer

X = np.array([[3,4],
              [1,2],
              [10,0]])

normalizer = Normalizer()
X_norm = normalizer.fit_transform(X)

df_norm = pd.DataFrame(
    np.hstack([X, X_norm]),
    columns=["x1","x2","x1(정규화 후)","x2(정규화 후)"]
)
df_norm

,x1,x2,x1(정규화 후),x2(정규화 후)
0,3.0,4.0,0.600000,0.800000
1,1.0,2.0,0.447214,0.894427
2,10.0,0.0,1.000000,0.000000


### 정규화와 코사인 유사도

- **코사인 유사도(cosine similarity)**는 두 벡터의 **방향(각도)**이 얼마나 비슷한지 보는 지표임
  - 같은 방향이면 1
  - 직각(관련 거의 없음)에 가까우면 0
  - 반대 방향이면 -1

- 벡터를 **L2 정규화**하면 길이가 1이 됨
- 이때는 **코사인 유사도 = 내적(dot product)**이 됨
  - (길이가 1이라서 분모가 사라지는 효과)

> 그래서 텍스트 벡터/임베딩 비교에서
> **정규화 + 내적(=코사인)** 조합이 자주 등장함

아래에서 직접 확인해볼 것. **세 값을 비교**함.

In [15]:
import numpy as np
from sklearn.preprocessing import Normalizer
from sklearn.metrics.pairwise import cosine_similarity

A = np.array([[3., 4.]])
B = np.array([[8., 6.]])   # 평행하지 않고 길이도 다른 벡터
                           # (평행하면 cosine이 항상 1이라 비교가 보이지 않음)

normer = Normalizer()
A_n = normer.fit_transform(A)
B_n = normer.transform(B)

print(f"원래 벡터의 내적        : {float(np.dot(A[0], B[0])):.4f}")
print(f"정규화된 벡터의 내적    : {float(np.dot(A_n[0], B_n[0])):.4f}")
print(f"cosine_similarity      : {float(cosine_similarity(A, B)[0, 0]):.4f}")

원래 벡터의 내적        : 48.0000
정규화된 벡터의 내적    : 0.9600
cosine_similarity      : 0.9600


### 왜 마지막 두 값이 같을까?

처음에는 벡터의 **길이와 방향이 둘 다** 내적에 영향을 줌.
길이를 전부 1로 맞추면 길이 차이는 사라지고 **방향만 비교하게 됨.**

> 그래서 정규화한 벡터에서는 **내적과 cosine similarity가 같아짐.**

### 문장 유사도 — CountVectorizer + 정규화 + 코사인 유사도

앞에서 배운 개념들을 조합하면 **문장이 얼마나 비슷한지**도 계산할 수 있음.

1. **CountVectorizer**: 문장 → 단어 빈도 벡터
2. **Normalizer**: 벡터 길이를 1로 맞춤 (방향만 남김)
3. **cosine_similarity**: 두 벡터의 방향이 같으면 1, 다르면 0에 가까움

> CountVectorizer는 문장을 숫자 벡터로 바꾸고,  
> 정규화와 코사인 유사도는 그 벡터의 **방향**이 같은지를 봄.
> 정확히는 지금 이 Bag-of-Words 공간에서 **단어 사용 패턴의 방향**이 얼마나 비슷한지임.
> "의미가 같다"와는 다름.

> **LLM의 임베딩도 결국 같은 아이디어**
> (문장을 벡터로 만들고, 벡터의 방향 유사도를 비교)
> 다만 CountVectorizer보다 훨씬 풍부한 의미를 담는 벡터를 사용

In [16]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity

texts = ["영화가 좋다", "영화가 진짜로 좋다"]

vectorizer = CountVectorizer()
X = vectorizer.fit_transform(texts)
print("단어 사전:", vectorizer.get_feature_names_out())

# cosine_similarity는 내부에서 정규화를 수행하므로 Normalizer를 따로 적용하지 않습니다.
sim = cosine_similarity(X[0], X[1])
print("코사인 유사도:", round(float(sim[0][0]), 4))
print("\n-> 단어가 겹친다는 뜻이지, 의미가 같다는 뜻은 아닙니다.")

단어 사전: ['영화가' '좋다' '진짜로']
코사인 유사도: 0.8165

-> 단어가 겹친다는 뜻이지, 의미가 같다는 뜻은 아닙니다.


### 스케일링 vs 정규화 — 언제 뭘 쓰나?

| 구분 | 스케일링 (Scaling) | 정규화 (Normalization) |
|------|-------------------|------------------------|
| 기준 | **열(Feature)** | **행(Sample)** |
| 바꾸는 것 | 피처의 단위/범위 | 벡터의 길이(L2 norm) |
| 목적 | **피처 간 scale 차이 조정** | **방향(패턴)만 남기기** |
| 주 사용처 | **스케일에 민감한 모델** | 텍스트/추천/임베딩(코사인) |


> **스케일에 민감한 모델(거리 기반, 일부 선형 모델)에서 주로 사용**
> 트리 기반 모델은 스케일 차이의 영향을 거의 받지 않음
> 정규화는 "방향"이 중요한 특수한 경우(텍스트/임베팅/추천/등)에만 사용함

---

### 지난 시간의 "뻥튀기" 기억나나?

2회차 첫 문제에서 342마리 전부로 학습하고 **같은 342마리로 채점**했더니 `1.0000`이 나왔음.
제대로 나눠서 채점하니 `0.9130`이었고.

> 그 **8.7%p가 거품**이었음.
> 같은 문제로 공부하고 같은 문제로 시험 보면 성적이 뻥튀기됨.

그래서 **평가용 데이터는 따로 남겨야 한다**고 했었음.

**오늘은 한 단계 더 감.**

> test를 따로 빼놓기만 하면 끝일까?
> **scaler를 맞출 때 test까지 본다면?**

미리 하나만 짚고 감. 지난 시간의 뻥튀기는 점수가 **진짜로 올라갔음.**
오늘 볼 문제는 **점수가 안 오를 수도 있음.** 그래도 문제임. 왜 그런지 곧 확인함.

---
## Part 4. 데이터 누수 (Data Leakage)

### 데이터 누수란?

- 훈련+시험 전체 데이터를 fit → **평가용으로 남겨둔 데이터의 정보**가 학습 과정에 미리 들어감
- 마치 학생이 **시험지를 미리 훔쳐보고 공부한 것**과 같음.

> 누수가 생기면 점수가 높아질 수도, 거의 달라지지 않을 수도 있음.
> 문제는 **평가용 데이터를 미리 썼기 때문에 그 평가를 믿을 수 없게 된다는 것**임.
>
> Pipeline은 전처리와 모델을 함께 묶어 이 실수를 줄이기 쉽게 해줌.
> 다만 Pipeline 밖에서 미리 fit한 처리나, 시험 결과를 보고 내린 선택까지 자동으로 막지는 못함.

### 데이터 누수가 왜 위험한가?

**누수가 발생하면:**
1. 실험 결과를 **신뢰할 수 없음**
2. 모델 선택이 **잘못될 수 있음**
3. 실제 성능을 **잘못 예상하거나**, 잘못된 모델/전처리를 선택할 수 있음

> **"전처리는 반드시 훈련셋으로만 fit!"**
> **"transform은 훈련·시험 둘 다!"**

규칙이 **두 개**임. 헷갈리면 안 됨.

| | 어디에 | 왜 |
|---|---|---|
| `fit` | **훈련 데이터에만** | 시험지 정보가 기준에 섞이면 안 됨 (**누수**) |
| `transform` | **훈련·시험 둘 다** | 같은 자로 재야 함 (**일관성**) |

> "test에 fit하지 마라"를 **"test는 건드리지 마라"**로 확장하면 안 됨.
> `transform`을 빼먹으면 모델은 **학습에 없던 크기의 숫자**를 받게 됨.
> 에러도 안 나고, 결과만 조용히 망가짐.

---

### 잠깐 — 실행 전에 예측해볼 것

시험 데이터까지 포함해서 scaler를 맞추면, **점수가 더 높아질까?**

> 예상 먼저 적고 아래 셀 실행할 것.

In [17]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, f1_score
from sklearn.pipeline import Pipeline

X, y = load_iris(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.3, stratify=y, random_state=42)

# (나쁜 예시) 전체 데이터로 fit
# 스탠다드스케일러(평균, 표준편차 계산)
# 시험 데이터(X_test) 데이터도 평균/표준편차가 반영이 됨
# 데이터 누수
scaler_bad = StandardScaler().fit(X)
# 위에서 계산된(시험 데이터가 포함된) 평균과 표준편차로, 학습/시험 데이터를 변환
X_train_bad = scaler_bad.transform(X_train)
X_test_bad  = scaler_bad.transform(X_test)
knn_bad = KNeighborsClassifier(n_neighbors=5).fit(X_train_bad, y_train)
pred_bad = knn_bad.predict(X_test_bad)
print(f"[누수 발생·KNN] Acc={accuracy_score(y_test, pred_bad):.3f}, F1(macro)={f1_score(y_test, pred_bad, average='macro'):.3f}")

[누수 발생·KNN] Acc=0.911, F1(macro)=0.910


In [18]:
# (Pipeline) 훈련에만 fit
pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("knn", KNeighborsClassifier(n_neighbors=5))
])
pipe.fit(X_train, y_train)
pred_pipe = pipe.predict(X_test)
print(f"[Pipeline·KNN]  Acc={accuracy_score(y_test, pred_pipe):.3f}, F1(macro)={f1_score(y_test, pred_pipe, average='macro'):.3f}")

[Pipeline·KNN]  Acc=0.911, F1(macro)=0.910


### Pipeline이란?

전처리 + 모델을 **한 줄로 묶어서** 실수를 막음.

```python
pipe = Pipeline([
    ("scaler", StandardScaler()),  # 전처리
    ("model", LogisticRegression()) # 모델
])

pipe.fit(X_train, y_train)   # 훈련셋으로 fit
pipe.predict(X_test)         # 시험셋은 transform만
```

> **train 데이터에 Pipeline을 fit하면, Pipeline 안의 전처리도 같은 train 데이터에서 함께 학습됨.**
> 교차검증에서 쓰면, 각 fold의 학습 데이터 안에서 전처리와 모델 학습이 함께 다시 수행됨.

**다만 Pipeline을 쓴다고 모든 누수가 사라지지는 않음.** 아래 두 가지는 막아주지 못함.

```python
X = PCA().fit_transform(X)   # Pipeline 밖에서 이미 누수
pipe.fit(X_train, y_train)
```

```python
# 시험 결과를 보고 기준값을 고르는 것도 Pipeline이 막아주지 않음 (3-2회차)
```

### 로지스틱 회귀(선형 모델)에서도 확인해 보기
- 선형 모델은 스케일링의 영향을 더 많이 받는 편임
- "누수 전처리 vs Pipeline"을 로지스틱 회귀에서도 비교

In [19]:
from sklearn.linear_model import LogisticRegression


# (나쁜 예시) 전체 데이터로 fit한 스케일러 사용
lr_bad = LogisticRegression(max_iter=5000, random_state=42)
lr_bad.fit(X_train_bad, y_train)
pred_lr_bad = lr_bad.predict(X_test_bad)

# (올바른 예시) Pipeline
pipe_lr = Pipeline([
    ("scaler", StandardScaler()),
    ("lr", LogisticRegression(max_iter=5000, random_state=42))
])
pipe_lr.fit(X_train, y_train)
pred_lr_pipe = pipe_lr.predict(X_test)

print(f"[누수 발생·LR]  Acc={accuracy_score(y_test, pred_lr_bad):.3f}, F1(macro)={f1_score(y_test, pred_lr_bad, average='macro'):.3f}")
print(f"[Pipeline·LR]   Acc={accuracy_score(y_test, pred_lr_pipe):.3f}, F1(macro)={f1_score(y_test, pred_lr_pipe, average='macro'):.3f}")

[누수 발생·LR]  Acc=0.911, F1(macro)=0.911
[Pipeline·LR]   Acc=0.911, F1(macro)=0.911


### 결과 해석

- Iris처럼 단순한 데이터는 차이가 작아 보일 수 있어도
  **누수 전처리 방식은 원칙적으로 금지**
- 점수가 얼마나 차이 나는지와 무관하게, **평가용 데이터의 정보를 미리 쓴 시점에서 그 평가는 공정하지 않음**
- **이 예제처럼 모든 전처리를 Pipeline 안에 넣고, Pipeline 자체를 훈련셋에 fit하면**
  전처리의 `fit` 범위를 올바르게 관리할 수 있음

> **핵심 요약**
> - 전처리는 반드시 **훈련셋으로만 fit**
> - 시험셋은 반드시 **처음 보는 데이터**처럼 다뤄야 함 (단, `transform`은 **반드시** 적용)
> - ColumnTransformer/Pipeline을 습관처럼 사용

### 교차검증으로 누수 vs Pipeline 비교

- train/test split은 한 번만 나누기 때문에 운에 따라 결과가 달라질 수 있음
- 교차검증(K-Fold CV)은 여러 분할에서 결과가 어떻게 변하는지 확인할 수 있음
- 이번에는 **누수 전처리 방식**과 **Pipeline 방식**을 교차검증으로 비교해볼 것

In [20]:
from sklearn.model_selection import cross_val_score, StratifiedKFold

X, y = load_iris(return_X_y=True)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# (나쁜 예시) 전체 데이터로 fit → transform 적용
scaler_bad = StandardScaler().fit(X)   # 전체 데이터 사용 → 누수
X_scaled_bad = scaler_bad.transform(X)

knn_bad = KNeighborsClassifier(n_neighbors=5)
scores_bad = cross_val_score(knn_bad, X_scaled_bad, y, cv=cv, scoring="accuracy")

# (올바른 예시) Pipeline (훈련셋만 fit, 시험셋은 transform만)
pipe_knn = Pipeline([
    ("scaler", StandardScaler()),
    ("knn", KNeighborsClassifier(n_neighbors=5))
])
scores_pipe = cross_val_score(pipe_knn, X, y, cv=cv, scoring="accuracy")

print("[누수 발생·KNN]   mean=%.3f ± %.3f" % (scores_bad.mean(), scores_bad.std()))
print("[Pipeline·KNN]    mean=%.3f ± %.3f" % (scores_pipe.mean(), scores_pipe.std()))

[누수 발생·KNN]   mean=0.967 ± 0.037


[Pipeline·KNN]    mean=0.973 ± 0.025


### 어? Pipeline이 더 높네

```
누수     0.9667
Pipeline 0.9733
```

예측과 반대임. 그러면 누수는 별 문제 없는 걸까?

> **아님.**
>
> 누수의 문제는 반드시 점수를 높인다는 게 아님.
> **평가용으로 남겨둔 데이터를 미리 봤기 때문에,
> 이 평가를 공정한 시험이라고 부를 수 없다는 것**이 문제임.

이번 Iris + KNN 실험에서는 두 방식의 평균 점수 차이가 아주 작았음.

> **왜 작았는지를 억지로 일반화하지 않음.**
> 결과가 비슷해도 평가 절차가 잘못됐다는 사실은 변하지 않음.

**그럼 절차만 어기면 결과는 항상 안전할까?** 다음에서 확인할 것.

### 로지스틱 회귀(Logistic Regression)로도 비교해보기

- 로지스틱은 스케일링의 영향을 더 많이 받는 모델임.
- 이번에는 로지스틱 회귀에서 **누수 vs Pipeline**을 교차검증으로 비교함.

In [21]:
from sklearn.linear_model import LogisticRegression

# (나쁜 예시) 누수 스케일링
lr_bad = LogisticRegression(max_iter=5000, random_state=42)
scores_lr_bad = cross_val_score(lr_bad, X_scaled_bad, y, cv=cv, scoring="accuracy")

# (올바른 예시) Pipeline
pipe_lr = Pipeline([
    ("scaler", StandardScaler()),
    ("lr", LogisticRegression(max_iter=5000, random_state=42))
])
scores_lr_pipe = cross_val_score(pipe_lr, X, y, cv=cv, scoring="accuracy")

print("[누수 발생·LR]   mean=%.3f ± %.3f" % (scores_lr_bad.mean(), scores_lr_bad.std()))
print("[Pipeline·LR]    mean=%.3f ± %.3f" % (scores_lr_pipe.mean(), scores_lr_pipe.std()))

[누수 발생·LR]   mean=0.953 ± 0.045
[Pipeline·LR]    mean=0.953 ± 0.045


### 조건을 바꿔서 다시 확인해볼 것

이번에는 **정답과 아무 관계도 없는 순수한 난수 데이터**를 씀.
`X`는 전부 무작위 숫자고, `y`는 동전 던지기임. 둘 사이에 패턴은 **존재하지 않음.**

> 그러니 정직하게 평가하면 정확도는 **0.5 근처**가 나와야 정상임.

In [22]:
import numpy as np
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.pipeline import Pipeline

n, p = 100, 5000
rng = np.random.RandomState(42)

X_rand = rng.normal(size=(n, p))      # 정답과 아무 관계 없는 순수 노이즈
y_rand = np.array([0, 1] * (n // 2))  # 정확히 50 : 50
rng.shuffle(y_rand)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# (나쁜 예시) 전체 데이터로 먼저 피처 선택 -> 그 다음 교차검증
selected = SelectKBest(f_classif, k=20).fit_transform(X_rand, y_rand)
score_leak = cross_val_score(LogisticRegression(max_iter=1000), selected, y_rand, cv=cv)

# (올바른 예시) Pipeline 안에서 fold마다 피처 선택
pipe_sel = Pipeline([
    ("select", SelectKBest(f_classif, k=20)),
    ("lr", LogisticRegression(max_iter=1000)),
])
score_pipe = cross_val_score(pipe_sel, X_rand, y_rand, cv=cv)

print(f"[누수 발생]  mean={score_leak.mean():.3f} +- {score_leak.std():.3f}")
print(f"[Pipeline ]  mean={score_pipe.mean():.3f} +- {score_pipe.std():.3f}")
print(f"[정답      ]  0.500  (X와 y는 아무 관계도 없으므로)")

[누수 발생]  mean=0.870 +- 0.081
[Pipeline ]  mean=0.520 +- 0.060
[정답      ]  0.500  (X와 y는 아무 관계도 없으므로)


### 존재하지 않는 패턴이 87%로 보임

`X`와 `y` 사이에는 **아무 관계도 없음.** 그런데 누수 방식은 0.87을 보고했음.

전체 데이터를 보고 "정답과 관련 있어 보이는" 컬럼 20개를 먼저 고른 순간,
평가용으로 남겨둔 데이터의 정답 정보가 이미 새어 들어간 것.

### 두 장면을 붙여서 읽을 것

| | 질문 | 결과 |
|---|---|---|
| Iris | 누수하면 점수가 오르나? | 아니오 (0.9667 vs 0.9733) |
| 난수 | 그럼 절차만 어기면 결과는 항상 안전한가? | 아니오 (0.870 vs 0.520, 정답 0.500) |

> **누수는 점수를 반드시 높이는 현상이 아님.**
> 어떤 경우에는 차이가 거의 없고, 어떤 경우에는 허구의 성능을 만들어냄.
>
> 공통점은 하나임.
> **평가용으로 남겨둔 데이터의 정보를 미리 썼다면, 그 점수를 공정한 평가로 믿을 수 없음.**

그래서 누수 여부는 **점수 차이를 보고 판단하는 게 아니라,
평가용 데이터의 정보를 학습 과정에서 썼는지를 보고 판단함.**

---
## 오늘의 정리

| 개념 | 핵심 | 코드 |
|------|------|------|
| 인코딩 | 글자→숫자, 순서 없으면 원-핫 | `OneHotEncoder()` |
| 스케일링 | 값 크기 차이 맞추기 | `StandardScaler()`, `MinMaxScaler()` |
| 정규화 | 샘플 길이=1, 방향만 중요할 때 | `Normalizer()` |
| 데이터 누수 | 시험 정보가 훈련에 들어가면 안 됨 | `Pipeline()` |

> **기억할 흐름**: 전처리(인코딩→스케일링) → Pipeline으로 묶기 → 교차검증

### 오늘의 관통 원칙

> **평가를 위해 남겨둔 데이터의 정보를 학습 과정에서 미리 사용하지 않는다.**

**이건 새로운 규칙이 아님.** 지난 시간에 이미 이렇게 나눴었음.

| 이름 | 역할 | 비유 |
|------|------|------|
| Train | 실제로 패턴을 학습하는 데이터 | 교과서 |
| Validation | 학습에 안 쓰고 채점만 하는 부분 | 모의고사 — 반복 사용 가능 |
| Test | 맨 마지막 딱 한 번용 | 수능 — 단 1회 |

> 배울 때는 **train**만 본다.
> 선택할 때는 **validation**(모의고사)을 본다.
> **test**(수능)는 모든 선택이 끝난 뒤 마지막에 한 번 본다.

지난 시간에 배운 이 규칙을 오늘은 **전처리**까지 넓힌 것.
3-2회차에서는 **기준값(threshold) 선택**까지 한 번 더 넓힙니다.

3-2회차에서 이 원칙을 기준값(threshold) 선택까지 확장함.

---
## 다음 시간 예고

- **평가지표 심화**: 정확도 90%를 믿어도 되나?
- **Confusion Matrix**: TP/FP/FN/TN 읽는 법
- **Precision vs Recall**: 스팸필터 vs 암진단
- **F1, ROC-AUC, PR-AP**: 불균형 데이터 평가

> 오늘 우리는 같은 질문을 두 번 만났음.
>
> - 모델이 숫자를 냈다 → 근거가 있었나? (0.5 / 0.5)
> - 점수가 나왔다 → 그 평가를 믿어도 되나? (누수)
>
> 3-2회차에서 세 번째를 만납니다. **정확도가 높다 → 필요한 일을 잘하고 있나?**